<a href="https://colab.research.google.com/github/14elias/LLM_Quantization/blob/main/quantization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# BUILD YOUR OWN MODEL QUANTIZER — FROM SCRATCH IN PYTORCH
# ============================================================
# Model: facebook/opt-125m
# Constraint: Run on a 4 GB RAM laptop (reduce size by ≥50%)
# Method: Pure PyTorch — NO external libraries (no bitsandbytes, no torch.ao)
# Framework: PURE ASYMMETRIC (AFFINE) QUANTIZATION
#   S = (r_max - r_min) / (q_max - q_min)
#   Z = round(-r_min / S) + q_min
#   q = clamp(round(W / S) + Z, q_min, q_max)
#   W_hat = S * (q - Z)
# ============================================================
# ── STEP 0: Install Dependencies (Uncomment if needed) ──────


# ── STEP 0: Install Dependencies ────────────────────────────

In [2]:
!pip install -q torch transformers accelerate safetensors tabulate matplotlib
import math
import time
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tabulate import tabulate
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "facebook/opt-125m"
print(f"Device: {DEVICE}  |  Model: {MODEL_ID}")


Device: cpu  |  Model: facebook/opt-125m


# ============================================================
# SECTION 1 — MATHEMATICAL QUANTIZATION KERNELS (Raw PyTorch)
# ============================================================

In [3]:
# ── 1.1 4-Bit Nibble Bit-Packing Helpers ───────────────────
def pack_int4(q: torch.Tensor) -> torch.Tensor:
    """Packs two 4-bit integers [-8, 7] into one torch.uint8 byte."""
    flat = q.reshape(-1)
    if flat.numel() % 2 != 0:
        flat = F.pad(flat, (0, 1))
    u4 = (flat.to(torch.int32) + 8).clamp(0, 15).to(torch.uint8)
    packed = u4[0::2] | (u4[1::2] << 4)
    packed_shape = list(q.shape)
    packed_shape[-1] = (packed_shape[-1] + 1) // 2
    return packed.reshape(packed_shape)
def unpack_int4(packed: torch.Tensor, orig_shape: torch.Size) -> torch.Tensor:
    """Unpacks torch.uint8 bytes back into signed int8 [-8, 7]."""
    flat = packed.reshape(-1)
    low  = (flat & 0x0F).to(torch.int8) - 8
    high = ((flat >> 4) & 0x0F).to(torch.int8) - 8
    out  = torch.empty(flat.numel() * 2, dtype=torch.int8, device=packed.device)
    out[0::2] = low
    out[1::2] = high
    return out[:math.prod(orig_shape)].reshape(orig_shape)
# ── 1.2 Asymmetric INT8 Quantization & Dequantization ──────
def quantize_asymmetric_int8(tensor: torch.Tensor, per_channel: bool = False):
    """
    Asymmetric 8-bit Quantization:
      S = (r_max - r_min) / (q_max - q_min)
      Z = round(-r_min / S) + q_min
      q = clamp(round(W / S) + Z, q_min, q_max)
    """
    q_min, q_max, eps = -128, 127, 1e-8
    if per_channel:
        r_min = tensor.amin(dim=1, keepdim=True)
        r_max = tensor.amax(dim=1, keepdim=True)
    else:
        r_min = tensor.min()
        r_max = tensor.max()
    scale = torch.clamp((r_max - r_min) / (q_max - q_min), min=eps)
    zero_point = torch.round(-r_min / scale) + q_min
    zero_point = torch.clamp(zero_point, q_min, q_max).to(torch.int8)
    q = torch.clamp(torch.round(tensor / scale) + zero_point, q_min, q_max).to(torch.int8)
    return q, scale.to(tensor.dtype), zero_point
def dequantize_asymmetric_int8(q: torch.Tensor, scale: torch.Tensor, zero_point: torch.Tensor):
    """Dequantize: W_hat = S * (q - Z)"""
    return (q.to(scale.dtype) - zero_point.to(scale.dtype)) * scale
# ── 1.3 Asymmetric INT4 Quantization (Per-Channel) ─────────
def quantize_asymmetric_int4(tensor: torch.Tensor, per_channel: bool = True):
    """Asymmetric 4-bit with 2-in-1 byte packing."""
    q_min, q_max, eps = -8, 7, 1e-8
    if per_channel:
        r_min = tensor.amin(dim=1, keepdim=True)
        r_max = tensor.amax(dim=1, keepdim=True)
    else:
        r_min = tensor.min()
        r_max = tensor.max()
    scale = torch.clamp((r_max - r_min) / (q_max - q_min), min=eps)
    zero_point = torch.round(-r_min / scale) + q_min
    zero_point = torch.clamp(zero_point, q_min, q_max).to(torch.int8)
    q = torch.clamp(torch.round(tensor / scale) + zero_point, q_min, q_max).to(torch.int8)
    return pack_int4(q), scale.to(tensor.dtype), zero_point, tensor.shape
def dequantize_asymmetric_int4(packed: torch.Tensor, scale: torch.Tensor, zero_point: torch.Tensor, orig_shape: torch.Size):
    """Unpack uint8 then dequantize: W_hat = S * (q - Z)"""
    q = unpack_int4(packed, orig_shape)
    return (q.to(scale.dtype) - zero_point.to(scale.dtype)) * scale
# ── 1.4 Asymmetric INT4 Grouped Quantization ───────────────
def quantize_asymmetric_int4_grouped(tensor: torch.Tensor, group_size: int = 64):
    """Divides channels into 64-element blocks; computes S and Z per block."""
    q_min, q_max, eps = -8, 7, 1e-8
    out_f, in_f = tensor.shape
    pad = (group_size - in_f % group_size) % group_size
    t = F.pad(tensor, (0, pad)) if pad else tensor
    ng = t.shape[1] // group_size
    grp = t.reshape(out_f, ng, group_size)
    r_min = grp.amin(dim=2, keepdim=True)
    r_max = grp.amax(dim=2, keepdim=True)
    scale = torch.clamp((r_max - r_min) / (q_max - q_min), min=eps)
    zero_point = torch.round(-r_min / scale) + q_min
    zero_point = torch.clamp(zero_point, q_min, q_max).to(torch.int8)
    q = torch.clamp(torch.round(grp / scale) + zero_point, q_min, q_max).to(torch.int8)
    q_flat = q.reshape(out_f, -1)[:, :in_f]
    return pack_int4(q_flat), scale.to(tensor.dtype), zero_point, tensor.shape
def dequantize_asymmetric_int4_grouped(packed: torch.Tensor, scale: torch.Tensor, zero_point: torch.Tensor, orig_shape: torch.Size, group_size: int = 64):
    out_f, in_f = orig_shape
    q = unpack_int4(packed, orig_shape)
    pad = (group_size - in_f % group_size) % group_size
    q_p = F.pad(q, (0, pad)) if pad else q
    ng = q_p.shape[1] // group_size
    grp = q_p.reshape(out_f, ng, group_size)
    deq = (grp.to(scale.dtype) - zero_point.to(scale.dtype)) * scale
    return deq.reshape(out_f, -1)[:, :in_f]


# ============================================================
# SECTION 2 — CUSTOM QuantizedLinear MODULE
# ============================================================

In [4]:
class QuantizedLinear(nn.Module):
    """Drop-in replacement for nn.Linear storing weights in Asymmetric quantized format."""
    SCHEMES = ("asym_int8_per_tensor", "asym_int8_per_channel", "asym_int4_per_channel", "asym_int4_grouped")
    def __init__(self, in_f, out_f, bias=True, scheme="asym_int8_per_channel", group_size=64):
        super().__init__()
        self.in_f, self.out_f = in_f, out_f
        self.scheme, self.group_size = scheme, group_size
        self.orig_shape = torch.Size([out_f, in_f])
        self.register_buffer("qweight", None)
        self.register_buffer("scale", None)
        self.register_buffer("zero_point", None)
        self.bias = nn.Parameter(torch.zeros(out_f)) if bias else None
    @classmethod
    def from_linear(cls, lin: nn.Linear, scheme="asym_int8_per_channel", group_size=64):
        obj = cls(lin.in_features, lin.out_features, bias=lin.bias is not None, scheme=scheme, group_size=group_size)
        w = lin.weight.data
        if scheme == "asym_int8_per_tensor":
            obj.qweight, obj.scale, obj.zero_point = quantize_asymmetric_int8(w, per_channel=False)
        elif scheme == "asym_int8_per_channel":
            obj.qweight, obj.scale, obj.zero_point = quantize_asymmetric_int8(w, per_channel=True)
        elif scheme == "asym_int4_per_channel":
            obj.qweight, obj.scale, obj.zero_point, _ = quantize_asymmetric_int4(w, per_channel=True)
        elif scheme == "asym_int4_grouped":
            obj.qweight, obj.scale, obj.zero_point, _ = quantize_asymmetric_int4_grouped(w, group_size=group_size)
        else:
            raise ValueError(f"Unknown scheme: {scheme}")
        if lin.bias is not None:
            obj.bias = nn.Parameter(lin.bias.data.clone())
        return obj
    def dequantize(self):
        if "int8" in self.scheme:
            return dequantize_asymmetric_int8(self.qweight, self.scale, self.zero_point)
        elif self.scheme == "asym_int4_per_channel":
            return dequantize_asymmetric_int4(self.qweight, self.scale, self.zero_point, self.orig_shape)
        else:
            return dequantize_asymmetric_int4_grouped(self.qweight, self.scale, self.zero_point, self.orig_shape, self.group_size)
    def forward(self, x):
        return F.linear(x, self.dequantize().to(x.dtype), self.bias.to(x.dtype) if self.bias is not None else None)


# ============================================================
# SECTION 3 — MODEL-LEVEL QUANTIZATION (Replace nn.Linear layers)
# ============================================================

In [5]:
TARGET_LAYERS = {"q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2", "gate_proj", "up_proj", "down_proj"}
SKIP_PREFIXES = ("lm_head", "embed_tokens")
def quantize_model(model: nn.Module, scheme="asym_int8_per_channel", group_size=64):
    """Recursively replaces target linear layers while preserving embeddings & lm_head in FP32."""
    def _replace(module, prefix=""):
        for name, child in list(module.named_children()):
            full = f"{prefix}.{name}" if prefix else name
            if any(full.startswith(s) for s in SKIP_PREFIXES):
                continue
            if isinstance(child, nn.Linear) and name in TARGET_LAYERS:
                setattr(module, name, QuantizedLinear.from_linear(child, scheme=scheme, group_size=group_size))
            else:
                _replace(child, full)
    _replace(model)
    return model
def model_size_mb(model: nn.Module) -> float:
    b = sum(p.numel() * p.element_size() for p in model.parameters())
    b += sum(buf.numel() * buf.element_size() for buf in model.buffers())
    return b / (1024 ** 2)

# ============================================================
# SECTION 4 — EVALUATION METRICS
# ============================================================

In [6]:
EVAL_TEXTS = [
    "Artificial intelligence is transforming industries by enabling machines to learn from experience.",
    "Quantum computing leverages the principles of superposition and entanglement.",
    "Deep neural networks have achieved remarkable success in natural language processing.",
    "The primary goal of model compression is to reduce memory and compute costs.",
    "Weight quantization maps floating-point values to low-bit integer representations.",
]
def perplexity(model, tokenizer, texts=EVAL_TEXTS, device=DEVICE):
    model.eval()
    loss_fn = nn.CrossEntropyLoss(reduction="sum")
    total_loss, total_tok = 0.0, 0
    with torch.no_grad():
        for txt in texts:
            ids = tokenizer(txt, return_tensors="pt").input_ids.to(device)
            logits = model(ids).logits
            sl = logits[..., :-1, :].contiguous().view(-1, logits.size(-1))
            tl = ids[..., 1:].contiguous().view(-1)
            total_loss += loss_fn(sl, tl).item()
            total_tok  += tl.numel()
    return float(torch.exp(torch.tensor(total_loss / max(1, total_tok))).item())
def gen_speed(model, tokenizer, device=DEVICE, prompt="The future of artificial intelligence is", n_tokens=25):
    model.eval()
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    with torch.no_grad():
        _ = model.generate(ids, max_new_tokens=2, do_sample=False)
        t0  = time.perf_counter()
        out = model.generate(ids, max_new_tokens=n_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        dt  = time.perf_counter() - t0
    gen  = out.shape[-1] - ids.shape[-1]
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return gen / dt if dt > 0 else 0.0, text
def weight_errors(fp32_model, quant_model):
    fd = dict(fp32_model.named_modules())
    mses, sims = [], []
    for name, qm in quant_model.named_modules():
        if hasattr(qm, "dequantize") and name in fd and hasattr(fd[name], "weight"):
            w  = fd[name].weight.data.float()
            wq = qm.dequantize().float()
            mses.append(((w - wq) ** 2).mean().item())
            sims.append(torch.cosine_similarity(w.flatten(), wq.flatten(), dim=0).item())
    return (sum(mses) / len(mses) if mses else 0.0, sum(sims) / len(sims) if sims else 1.0)


# ============================================================
# SECTION 5 — RUN BENCHMARKS ON ALL CONFIGURATIONS
# ============================================================

In [7]:
CONFIGS = [
    ("Baseline (FP32)",               None,                     64),
    ("Asym INT8 Per-Tensor",          "asym_int8_per_tensor",   64),
    ("Asym INT8 Per-Channel [WINNER]","asym_int8_per_channel",  64),
    ("Asym INT4 Per-Channel",         "asym_int4_per_channel",  64),
    ("Asym INT4 Grouped (G=64)",       "asym_int4_grouped",      64),
]
print(f"\nLoading tokenizer for {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Loading FP32 baseline ...")
fp32_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to(DEVICE)
base_size = model_size_mb(fp32_model)
base_ppl  = perplexity(fp32_model, tokenizer)
base_spd, base_txt = gen_speed(fp32_model, tokenizer)
results = {}
results["Baseline (FP32)"] = dict(
    size=base_size, reduction=0.0, ppl=base_ppl, ppl_shift=0.0,
    mse=0.0, cos=1.0, speed=base_spd, text=base_txt
)
print(f"  Baseline: {base_size:.1f} MB | PPL {base_ppl:.2f} | {base_spd:.1f} tok/s\n")
for label, scheme, gs in CONFIGS[1:]:
    print(f"--- {label} ---")
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to(DEVICE)
    quantize_model(model, scheme=scheme, group_size=gs)
    sz   = model_size_mb(model)
    red  = (base_size - sz) / base_size * 100
    ppl  = perplexity(model, tokenizer)
    delt = (ppl - base_ppl) / base_ppl * 100
    mse, cos = weight_errors(fp32_model, model)
    spd, txt = gen_speed(model, tokenizer)
    results[label] = dict(size=sz, reduction=red, ppl=ppl, ppl_shift=delt, mse=mse, cos=cos, speed=spd, text=txt)
    print(f"  {sz:.1f} MB  (-{red:.1f}%)  PPL {ppl:.2f}  ({delt:+.1f}%)  MSE {mse:.2e}  cos {cos:.5f}  {spd:.1f} tok/s\n")



Loading tokenizer for facebook/opt-125m ...


config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

  235.1 MB  (-50.8%)  PPL 78.07  (-0.5%)  MSE 1.50e-07  cos 0.99999  3.2 tok/s

--- Asym INT4 Per-Channel ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

  194.6 MB  (-59.3%)  PPL 123.71  (+57.6%)  MSE 4.26e-05  cos 0.98809  5.6 tok/s

--- Asym INT4 Grouped (G=64) ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

  200.6 MB  (-58.0%)  PPL 83.71  (+6.7%)  MSE 1.59e-05  cos 0.99541  5.3 tok/s



# ════════════════════════════════════════════════════════════════
# SECTION 6 — RESULTS TABLE
# ════════════════════════════════════════════════════════════════

In [8]:
rows = []
for name, d in results.items():
    rows.append([
        name,
        f"{d['size']:.1f} MB",
        f"{d['reduction']:.1f}%",
        f"{d['ppl']:.2f}",
        f"{d['ppl_shift']:+.1f}%",
        f"{d['mse']:.2e}",
        f"{d['cos']:.5f}",
        f"{d['speed']:.1f}",
    ])
headers = ["Configuration", "Size", "Size↓", "PPL", "PPL shift", "MSE", "Cos Sim", "tok/s"]
print("\n" + "=" * 95)
print("RESULTS TABLE (All configurations evaluated purely with Asymmetric S and Z formula)")
print("=" * 95)
print(tabulate(rows, headers=headers, tablefmt="github"))
print("=" * 95)



RESULTS TABLE (All configurations evaluated purely with Asymmetric S and Z formula)
| Configuration                  | Size     | Size↓   |    PPL | PPL shift   |      MSE |   Cos Sim |   tok/s |
|--------------------------------|----------|---------|--------|-------------|----------|-----------|---------|
| Baseline (FP32)                | 477.8 MB | 0.0%    |  78.49 | +0.0%       | 0        |   1       |    25.1 |
| Asym INT8 Per-Tensor           | 234.8 MB | 50.9%   |  78.34 | -0.2%       | 2.01e-06 |   0.9989  |     3.4 |
| Asym INT8 Per-Channel [WINNER] | 235.1 MB | 50.8%   |  78.07 | -0.5%       | 1.5e-07  |   0.99999 |     3.2 |
| Asym INT4 Per-Channel          | 194.6 MB | 59.3%   | 123.71 | +57.6%      | 4.26e-05 |   0.98809 |     5.6 |
| Asym INT4 Grouped (G=64)       | 200.6 MB | 58.0%   |  83.71 | +6.7%       | 1.59e-05 |   0.99541 |     5.3 |



# ════════════════════════════════════════════════════════════════
# SECTION 7 — TEXT GENERATION SAMPLES
# ════════════════════════════════════════════════════════════════

In [9]:
print("\n" + "=" * 80)
print("SAMPLE GENERATION (Prompt: 'The future of artificial intelligence is')")
print("=" * 80)
for name, d in results.items():
    print(f"\n[{name}]:")
    print(d["text"])
print("=" * 80)


SAMPLE GENERATION (Prompt: 'The future of artificial intelligence is')

[Baseline (FP32)]:
The future of artificial intelligence is in the hands of the people.

The future of artificial intelligence is in the hands of the people.

The

[Asym INT8 Per-Tensor]:
The future of artificial intelligence is in the hands of the people.

The future of artificial intelligence is in the hands of the people.

The

[Asym INT8 Per-Channel [WINNER]]:
The future of artificial intelligence is in the hands of the people.

The future of artificial intelligence is in the hands of the people.

The

[Asym INT4 Per-Channel]:
The future of artificial intelligence is here.

The future of artificial intelligence is here.

The future of artificial intelligence is here.

The

[Asym INT4 Grouped (G=64)]:
The future of artificial intelligence is in the hands of the people.

The future of artificial intelligence is in the hands of the people.

The


# ════════════════════════════════════════════════════════════════
# SECTION 8 — PLOT
# ════════════════════════════════════════════════════════════════

In [10]:

names  = list(results.keys())
sizes  = [results[k]["size"]  for k in names]
ppls   = [results[k]["ppl"]   for k in names]
mses   = [results[k]["mse"]   for k in names]
spds   = [results[k]["speed"] for k in names]
colors = ["#4A90E2", "#9B59B6", "#27AE60", "#E74C3C", "#F39C12"]
fig, axs = plt.subplots(2, 2, figsize=(14, 9))
plt.suptitle(f"Model Quantizer: Asymmetric Trade-off Analysis ({MODEL_ID})", fontsize=14, fontweight="bold")
for ax, vals, title, ylabel in [
    (axs[0, 0], sizes, "Model Footprint (MB) [↓ Better]",      "Megabytes"),
    (axs[0, 1], ppls,  "Perplexity (PPL) [↓ Better]",          "PPL"),
    (axs[1, 0], mses,  "Weight Reconstruction MSE [↓ Better]", "MSE"),
    (axs[1, 1], spds,  "Generation Speed (tok/s) [↑ Better]",  "tok/s"),
]:
    bars = ax.bar(names, vals, color=colors[:len(names)])
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=9)
    ax.tick_params(axis="x", rotation=25, labelsize=7.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.02,
                f"{v:.1e}" if v < 0.01 else f"{v:.1f}",
                ha="center", va="bottom", fontsize=7.5)
plt.tight_layout()
plt.savefig("quantizer_results.png", dpi=150, bbox_inches="tight")
print("\nChart saved → quantizer_results.png")



Chart saved → quantizer_results.png


# ════════════════════════════════════════════════════════════════
# SECTION 9 — EXPORT WINNER (Asym INT8 Per-Channel)
# ════════════════════════════════════════════════════════════════

In [11]:
OUTPUT_DIR = "./quantized_opt125m_asym_int8"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\nExporting winner (Asymmetric INT8 Per-Channel) to '{OUTPUT_DIR}' ...")
winner = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to(DEVICE)
quantize_model(winner, scheme="asym_int8_per_channel")
tokenizer.save_pretrained(OUTPUT_DIR)
AutoConfig.from_pretrained(MODEL_ID).save_pretrained(OUTPUT_DIR)
torch.save(winner.state_dict(), os.path.join(OUTPUT_DIR, "pytorch_model.bin"))
meta = {
    "base_model": MODEL_ID,
    "quant_type": "asymmetric_int8_per_channel",
    "formula": "S = (r_max - r_min) / (q_max - q_min), Z = round(-r_min / S) + q_min",
    "size_mb": round(model_size_mb(winner), 2),
    "quantized_layers": list(TARGET_LAYERS),
    "preserved_layers": list(SKIP_PREFIXES),
}
with open(os.path.join(OUTPUT_DIR, "quant_config.json"), "w") as f:
    json.dump(meta, f, indent=2)
print(f"  Files written to '{OUTPUT_DIR}'")
print(f"  Quantized model size: {meta['size_mb']} MB (FP32 baseline was {base_size:.1f} MB)\n")



Exporting winner (Asymmetric INT8 Per-Channel) to './quantized_opt125m_asym_int8' ...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

  Files written to './quantized_opt125m_asym_int8'
  Quantized model size: 235.15 MB (FP32 baseline was 477.8 MB)



# ════════════════════════════════════════════════════════════════
# SECTION 10 — PUSH TO HUGGING FACE HUB
# ════════════════════════════════════════════════════════════════

In [13]:
from huggingface_hub import HfApi

HF_TOKEN   = "hf_token"
HF_REPO_ID = "1elion/opt-125m-asym-int8"

api = HfApi(token=HF_TOKEN)

# 1. Create the repository on Hugging Face (if it doesn't exist)
api.create_repo(repo_id=HF_REPO_ID, exist_ok=True)

# 2. Upload the quantized model folder
api.upload_folder(
    folder_path="./quantized_opt125m_asym_int8",
    repo_id=HF_REPO_ID
)

print(f"Successfully uploaded to: https://huggingface.co/{HF_REPO_ID}")

Successfully uploaded to: https://huggingface.co/1elion/opt-125m-asym-int8
